# Intro to Cognee - building a Personal Memory Assistant

## Why personal memory?

Personal attention is fragmented by default. A tiny obligation can arrive in a Slack mention, a family chat, an email thread, or a follow-up buried under a reply. The hard part is not just storing the messages. The hard part is noticing which ones still expect something from you.

A personal memory assistant should answer questions like:

- Who tagged me and still needs a reply?
- What did I already handle?
- What did I answer, but still need to finish?
- What is due by EOD, EOW, or a specific posted deadline?

This notebook builds that foundation with Cognee. We ingest a small personal corpus for Veljko: Slack-style channel exports and email-thread exports. Then we ask the graph for an EOD/EOW reminder brief.

The notebook follows a simple memory-assistant loop:

1. **Load data** - `remember()` ingests text and builds the graph in one call
2. **Recall** - ask natural-language questions over the personal memory
3. **Node sets** - tag Slack and email sources for scoping
4. **Custom graph model** - extract typed obligations and response state
5. **Add another source** - email joins the same graph as Slack
6. **Life-assistant brief** - ask for missed replies, handled items, and open follow-up


## The API (Cognee 1.x)

The modern surface is three async functions:

```python
await cognee.remember(text, dataset_name=...)   # 1. ingest + build the graph
await cognee.recall(query)                       # 2. ask questions
await cognee.improve(dataset=...)                # 3. refine the graph from feedback/use
```

`remember()` replaces the old two-step `add()` + `cognify()` flow for everyday use. It ingests the text, extracts graph structure, and runs the default self-improvement pass.

For a personal assistant, that means every normalized source - Slack, email, notes, calendar summaries - can become memory with the same call. The source matters less than the shape: clear speaker, timestamp, and message text.


## Setup

The setup cell below puts a placeholder `LLM_API_KEY` into `os.environ`. Replace `"your-api-key"` with a real key before running the notebook.

> Run once from the repo root: `uv pip install -e . jupyterlab ipykernel`


In [ ]:
import os, re
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
os.environ.setdefault("LLM_API_KEY", "your-api-key")
os.environ.setdefault("CACHING", "true")

import cognee
from cognee.modules.users.methods import get_default_user
from cognee.modules.data.methods import get_authorized_existing_datasets
from cognee.context_global_variables import set_database_global_context_variables

# Works whether the kernel starts in the repo root or a nested notebooks directory.
ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "sample_data" / "personal_memory").exists():
        REPO = candidate
        break
else:
    raise FileNotFoundError("Could not find sample_data/personal_memory from the current notebook directory")

DATA = REPO / "sample_data" / "personal_memory"
DATASET = "personal_memory"
TARGET_PERSON = "veljko@topoteretes.com"
TARGET_MENTION = "@veljko@topoteretes.com"

async def reset():
    """Wipe local Cognee state so the notebook can be re-run cleanly."""
    await cognee.prune.prune_data()
    await cognee.prune.prune_system(metadata=True)

def answer(results):
    """Pull a readable answer out of a recall() result list."""
    for r in results:
        for attr in ("text", "answer", "content"):
            value = getattr(r, attr, None)
            if value:
                return value
    return str(results[0]) if results else "<no answer>"

async def show_graph(filename, dataset, height=760):
    """Render a dataset-scoped graph to HTML, with open/download controls."""
    user = await get_default_user()
    datasets = await get_authorized_existing_datasets([dataset], "read", user)

    if not datasets:
        raise ValueError(f"Dataset not found or not readable: {dataset}")

    dataset_obj = datasets[0]
    owner_id = getattr(dataset_obj, "owner_id", None) or user.id

    out = ROOT / filename
    async with set_database_global_context_variables(dataset_obj.id, owner_id):
        await cognee.visualize_graph(str(out))

    from html import escape
    from IPython.display import HTML, IFrame, display

    href = escape(filename, quote=True)
    controls = (
        '<div style="display:flex;gap:8px;margin:8px 0 10px;align-items:center">'
        f'<a href="{href}" target="_blank" rel="noopener" '
        'style="padding:6px 10px;border:1px solid #ccc;border-radius:6px;text-decoration:none">'
        'Open full screen</a>'
        f'<a href="{href}" download '
        'style="padding:6px 10px;border:1px solid #ccc;border-radius:6px;text-decoration:none">'
        'Download HTML</a>'
        '</div>'
    )
    display(HTML(controls))
    return IFrame(src=filename, width="100%", height=height)


print("Data directory:", DATA)
print("Slack files:", [p.name for p in sorted(DATA.glob("slack_*.txt"))])
print("Email files:", [p.name for p in sorted(DATA.glob("email_*.txt"))])

## 1. Load Slack messages with `remember()`

The Slack files use a lightweight export format: one channel transcript per file, with one line per message.

```text
# #channel-name: short thread title
[person@example.com, 2026-06-11T09:12] message text
```

In a real assistant, these would come from the Slack API. For the notebook, plain text keeps the data easy to inspect.


In [ ]:
slack_files = sorted(DATA.glob("slack_*.txt"))
print(slack_files[0].read_text())


In [ ]:
await reset()

slack_files = sorted(DATA.glob("slack_*.txt"))
for f in slack_files:
    result = await cognee.remember(
        f.read_text(),
        dataset_name=DATASET,
        node_set=["source:slack", f"file:{f.stem}"],
    )
    print("remembered", f.name, "- status:", result.to_dict().get("status"))


### Graph: Slack-only default extraction

This renders the first dataset while it still exists. The next section resets local state to keep the node-set demo isolated.

In [ ]:
await show_graph("personal_memory_slack_default_graph.html", DATASET)


## 2. Recall - ask for missed replies

Now ask the graph like a personal assistant. The key distinction is not just "was Veljko mentioned?" but whether the thread shows a response or completion.


In [ ]:
results = await cognee.recall(
    "Which Slack messages tagged Veljko and still need his response or action by EOD 2026-06-11? Separate missed replies from items where he replied but more work is needed.",
    datasets=[DATASET],
)
print(answer(results))


In [ ]:
# Peek at the retrieved context behind the reminder answer.
ctx = await cognee.recall(
    "Veljko tagged EOD missed response open action",
    datasets=[DATASET],
    only_context=True,
    top_k=5,
)
for c in ctx:
    print("-", str(getattr(c, "content", c))[:300])


## 3. Node sets - separate Slack from email

Node sets are tags attached at ingest time. Here we tag each source by channel/file and by broader source type, so the assistant can later answer from only Slack, only email, or the whole personal memory.

```python
await cognee.remember(text, dataset_name=DATASET, node_set=["source:slack"])
await cognee.remember(text, dataset_name=DATASET, node_set=["source:email"])
```


In [ ]:
NS_DATASET = "personal_memory_node_sets"
await reset()

for f in sorted(DATA.glob("slack_*.txt")):
    await cognee.remember(f.read_text(), dataset_name=NS_DATASET, node_set=["source:slack", f"file:{f.stem}"])
for f in sorted(DATA.glob("email_*.txt")):
    await cognee.remember(f.read_text(), dataset_name=NS_DATASET, node_set=["source:email", f"file:{f.stem}"])

print("tagged Slack and email into separate node sets")


In [ ]:
slack_only = await cognee.recall(
    "What Slack obligations are still open for Veljko?",
    datasets=[NS_DATASET],
    node_name=["source:slack"],
    scope=["graph"],
)
print(answer(slack_only))


### Graph: Slack and email node-set dataset

This dataset is tagged by source (`source:slack`, `source:email`) and by file. It shows how the same corpus can be organized for source scoping.

In [ ]:
await show_graph("personal_memory_node_sets_graph.html", NS_DATASET)


## 4. Custom graph model - extract obligations

Default extraction can answer questions, but a life assistant benefits from a typed schema. We care about people, messages, and obligations:

- who asked
- who owns the next action
- what the deadline is
- whether the state is `missed_response`, `open_followup`, or `handled`

The custom graph model below defines those concepts directly as `Person`, `Message`, `Obligation`, and `MemoryThread`. The graph visualization in this section should therefore contain these newly defined model nodes, rather than only Cognee's default `Entity` / `EntityType` nodes.

The extraction prompt is intentionally strict: acknowledging a request is not the same as completing it.


In [ ]:
from typing import Optional
from cognee.low_level import DataPoint

class Person(DataPoint):
    email: str
    name: str
    metadata: dict = {"index_fields": ["email", "name"], "identity_fields": ["email"]}

class Obligation(DataPoint):
    summary: str
    owner: Person
    requested_by: Person
    deadline: str
    status: str
    next_action: str
    evidence: str
    source: str
    metadata: dict = {"index_fields": ["summary", "deadline", "status", "next_action"]}

class Message(DataPoint):
    speaker: Person
    text: str
    sent_at: str
    mentions: Optional[list[Person]] = None
    obligations: Optional[list[Obligation]] = None
    metadata: dict = {"index_fields": ["text", "sent_at"]}

class MemoryThread(DataPoint):
    source: str
    title: str
    participants: list[Person]
    messages: list[Message]
    obligations: Optional[list[Obligation]] = None
    metadata: dict = {"index_fields": ["source", "title"]}


In [ ]:
EXTRACTION_PROMPT = f"""Extract a personal memory graph from this Slack or email thread.

Target person: {TARGET_PERSON}

Use the provided MemoryThread graph model with Person, Message, and Obligation nodes.
Keep source evidence precise and short.

For every request, question, or deadline directed at the target person, create an Obligation.
Set status as one of:
- missed_response: the target person was asked or tagged and no later target-person reply appears.
- open_followup: the target person replied or acknowledged, but the text does not show completion, or a later message says the work is still missing.
- handled: the target person replied and the thread shows the request was completed or no more action is needed.

Deadline rules:
- Preserve explicit dates and times when present.
- Preserve EOD and EOW wording together with any stated weekday or date.
- Preserve explicit dates such as Thursday 2026-06-11 and Friday 2026-06-12.

Do not mark acknowledgement as handled unless completion is explicit.
Prefer exact source text over summaries."""

TYPED_DATASET = "personal_memory_typed"
await reset()

for f in sorted(DATA.glob("slack_*.txt")):
    await cognee.remember(
        f.read_text(),
        dataset_name=TYPED_DATASET,
        node_set=["source:slack", f"file:{f.stem}"],
        graph_model=MemoryThread,
        custom_prompt=EXTRACTION_PROMPT,
    )

print("typed Slack memory graph built")


### Graph: Custom obligation model

This graph is built with the custom `MemoryThread` schema. Look for nodes shaped by the notebook's model definitions, especially `Person`, `Message`, `Obligation`, and `MemoryThread`, instead of the generic default extraction types.

In [ ]:
await show_graph("personal_memory_custom_model_slack_graph.html", TYPED_DATASET)


In [ ]:
typed_slack = await cognee.recall(
    "List Veljko's Slack obligations grouped by status: missed_response, open_followup, handled. Include deadline and source evidence.",
    datasets=[TYPED_DATASET],
)
print(answer(typed_slack))


## 5. Add email threads to the same graph

Now add email as a second source. The format is different from Slack, but it is still plain text with people, timestamps, and messages. We use the same `remember()` call and the same typed schema.


In [ ]:
email_files = sorted(DATA.glob("email_*.txt"))
print(email_files[0].read_text())


In [ ]:
for f in sorted(DATA.glob("email_*.txt")):
    await cognee.remember(
        f.read_text(),
        dataset_name=TYPED_DATASET,
        node_set=["source:email", f"file:{f.stem}"],
        graph_model=MemoryThread,
        custom_prompt=EXTRACTION_PROMPT,
    )
    print("remembered", f.name)


## 6. Life-assistant brief

This is the payoff: one typed obligation graph spanning Slack and email. Ask it for a reminder brief that a personal assistant could send at the end of the day or end of the week.


In [ ]:
brief = await cognee.recall(
    "Create Veljko's reminder brief for Thursday 2026-06-11. Include: (1) missed replies due by EOD, (2) items he answered but still needs to finish, (3) items already handled and safe to ignore. Include source and deadline for each item.",
    datasets=[TYPED_DATASET],
)
print(answer(brief))


In [ ]:
eow = await cognee.recall(
    "What should Veljko remember before EOW Friday 2026-06-12? Focus on unresolved obligations across Slack and email.",
    datasets=[TYPED_DATASET],
)
print(answer(eow))


## Visualize The Result In The Notebook

Render the final `TYPED_DATASET` as an embedded interactive graph before opening the local UI. This shows the personal-memory graph produced by the SDK flow, including the notebook-defined model types (`MemoryThread`, `Message`, `Person`, `Obligation`) rather than only Cognee default `Entity` / `EntityType` nodes.


In [ ]:
await show_graph("personal_memory_typed_graph.html", TYPED_DATASET)


## Start Local UI And Backend

Start Cognee local UI from the imported `cognee` library. The backend uses this notebook kernel environment, so it sees the same local Cognee settings and database paths as the tutorial.


In [ ]:
import time
import urllib.request

import cognee

COGNEE_BACKEND_URL = "http://localhost:8000"
COGNEE_FRONTEND_URL = "http://localhost:3000"

cognee_ui_pids = []

def remember_cognee_ui_pid(pid_or_tuple):
    pid = pid_or_tuple[0] if isinstance(pid_or_tuple, tuple) else pid_or_tuple
    cognee_ui_pids.append(pid)
    print(f"Started Cognee UI process pid={pid}")

def url_ready(url: str, timeout_seconds: float = 1.5) -> bool:
    try:
        with urllib.request.urlopen(url, timeout=timeout_seconds) as response:
            return 200 <= response.status < 500
    except Exception:
        return False

def wait_for_url(name: str, url: str, timeout_seconds: int = 90) -> None:
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        if url_ready(url):
            print(f"{name} ready: {url}")
            return
        time.sleep(1)
    print(f"{name} did not answer within {timeout_seconds}s: {url}")

cognee_frontend_process = cognee.start_ui(
    pid_callback=remember_cognee_ui_pid,
    port=3000,
    open_browser=False,
    auto_download=True,
    start_backend=True,
    backend_port=8000,
    start_mcp=False,
)

if cognee_frontend_process is None:
    raise RuntimeError("Cognee UI did not start. Check the cell output above for npm/backend errors.")

wait_for_url("Cognee backend", f"{COGNEE_BACKEND_URL}/health")
wait_for_url("Cognee frontend", COGNEE_FRONTEND_URL, timeout_seconds=120)

print(f"Open the local UI: {COGNEE_FRONTEND_URL}")


## Terminate Local UI And Backend

Run the next cell when you are done.


In [ ]:
import os
import signal
import time

def process_is_running(pid: int) -> bool:
    try:
        os.kill(pid, 0)
        return True
    except OSError:
        return False

def stop_process_group(pid: int, timeout_seconds: int = 15) -> None:
    if not process_is_running(pid):
        print(f"Process pid={pid} is already stopped.")
        return

    print(f"Stopping Cognee UI process group pid={pid}")
    try:
        os.killpg(pid, signal.SIGTERM)
    except ProcessLookupError:
        print(f"Process group pid={pid} is already stopped.")
        return

    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        if not process_is_running(pid):
            print(f"Stopped pid={pid}.")
            return
        time.sleep(0.5)

    print(f"pid={pid} did not stop after SIGTERM; sending SIGKILL.")
    try:
        os.killpg(pid, signal.SIGKILL)
    except ProcessLookupError:
        pass

for pid in reversed(globals().get("cognee_ui_pids", [])):
    stop_process_group(pid)

cognee_ui_pids = []
cognee_frontend_process = None
